# Demographic and admission features

Use the existing adult ICU cohort without filtering, reordering or expanding it. Output exactly `ICUSTAY_ID`, `age`, `gender`, `admission_type`, `admission_location` to `data/processed/demographic_features.parquet`.

## Sources and timing

- Existing `adult_icu_cohort_first24h.parquet`: ICUSTAY_ID, SUBJECT_ID, HADM_ID, INTIME.
- PATIENTS: **only SUBJECT_ID, DOB, GENDER**; left join on SUBJECT_ID with many-to-one validation.
- ADMISSIONS: **only SUBJECT_ID, HADM_ID, ADMISSION_TYPE, ADMISSION_LOCATION**; left join on both hospital-admission and patient identifiers with many-to-one validation.

No death, discharge, outcome, insurance, marital status, ethnicity, religion or language columns are read. Admission attributes are treated as information available at hospital admission; no later measurement or treatment information is used. The retrospective source does not provide field-level revision history.

## Age policy

Compute continuous age at ICU entry as `(INTIME - DOB).total_seconds() / (365.25 * 86400)`. Use microsecond datetime resolution to avoid overflowing nanosecond timedeltas when DOB is shifted by centuries. Do not round age before applying the adult-cohort checks.

[MIMIC-III PATIENTS documentation](https://mimic.mit.edu/docs/iii/tables/patients.html) explains that DOB is shifted for older patients, yielding ages near 300 years. Following the existing project's top-coding convention, cap calculated ages at **90**. The value 90 represents the upper age group, including anonymized older patients; it is not a recovered exact age. No age-group column is added. Missing source values remain missing; no imputation or category encoding is applied. Original category strings, including explicit unknown categories, are preserved.


In [1]:
import pandas as pd
from pathlib import Path
import hashlib
import numpy as np

data_path = Path('/mnt/c/Users/vetts/Downloads/MimicIII/mimic-iii-clinical-database-1.4')
project_root = Path.cwd().resolve()
if project_root.name == 'notebooks':
    project_root = project_root.parent
processed = project_root / 'data/processed'
output_path = processed / 'demographic_features.parquet'

def source_path(table):
    path = data_path / (table + '.csv')
    return path if path.is_file() else path / (table + '.csv')

def sha256(path):
    with path.open('rb') as handle:
        return hashlib.file_digest(handle, 'sha256').hexdigest()

protected = sorted(p for p in processed.iterdir() if p.is_file() and p != output_path)
protected += sorted(p for p in (project_root/'notebooks').iterdir() if p.is_file() and p.name != '05_demographics.ipynb')
protected_before = {str(p):sha256(p) for p in protected}
cohort = pd.read_parquet(processed/'adult_icu_cohort_first24h.parquet', columns=['ICUSTAY_ID','SUBJECT_ID','HADM_ID','INTIME'])
assert len(cohort) == cohort.ICUSTAY_ID.nunique() == 45253
assert not cohort.isna().any().any()
patients = pd.read_csv(source_path('PATIENTS'), usecols=['SUBJECT_ID','DOB','GENDER'])
admissions = pd.read_csv(source_path('ADMISSIONS'), usecols=['SUBJECT_ID','HADM_ID','ADMISSION_TYPE','ADMISSION_LOCATION'])
assert patients.SUBJECT_ID.is_unique
assert admissions.HADM_ID.is_unique
assert not admissions.duplicated(['SUBJECT_ID','HADM_ID']).any()
merged = cohort.merge(patients,on='SUBJECT_ID',how='left',sort=False,validate='many_to_one',indicator='_patient_match')
assert merged['_patient_match'].eq('both').all(), 'Unmatched patient identifier'
merged = merged.drop(columns='_patient_match').merge(admissions,on=['SUBJECT_ID','HADM_ID'],how='left',sort=False,validate='many_to_one',indicator='_admission_match')
assert merged['_admission_match'].eq('both').all(), 'Unmatched patient/admission pair'
assert merged.ICUSTAY_ID.equals(cohort.ICUSTAY_ID)
print('Cohort:',len(cohort),'unique ICU stays')
print('All patient and hospital-admission matches validated; no rows dropped or expanded.')


Cohort: 45253 unique ICU stays
All patient and hospital-admission matches validated; no rows dropped or expanded.


In [2]:
def age_at_icu_entry(intime, dob):
    intime = pd.to_datetime(intime, errors='raise').astype('datetime64[us]')
    dob = pd.to_datetime(dob, errors='raise').astype('datetime64[us]')
    return (intime-dob).dt.total_seconds() / (365.25*86400)

# Check ordinary, anonymized and missing DOB behavior independently.
sample_age = age_at_icu_entry(pd.Series(['2100-01-01']*3), pd.Series(['2060-01-01','1800-01-01',None]))
assert abs(sample_age.iloc[0]-40) < 0.01
assert sample_age.iloc[1] > 250 and sample_age.clip(upper=90).iloc[1] == 90
assert pd.isna(sample_age.iloc[2])
age_raw = age_at_icu_entry(merged.INTIME,merged.DOB)
assert np.isfinite(age_raw.dropna()).all()
assert age_raw.dropna().ge(18).all(), 'Unexpected age below 18; do not modify the cohort'
features = merged[['ICUSTAY_ID']].copy()
features['age'] = age_raw.clip(upper=90)
features['gender'] = merged.GENDER
features['admission_type'] = merged.ADMISSION_TYPE
features['admission_location'] = merged.ADMISSION_LOCATION
print('Age calculation: (ICU INTIME - DOB) in seconds / (365.25 * 86400)')
print('Upper cap: 90, a group representation, not an exact recovered age')
print('Raw ages above 120 (anonymization diagnostic):', int(age_raw.gt(120).sum()))
print('Raw ages above 90 capped:', int(age_raw.gt(90).sum()))
print('Age min / median / max:', features.age.min(), features.age.median(), features.age.max())
print('\nMissing values:')
print(features.isna().sum().to_string())
for column in ['gender','admission_type','admission_location']:
    print('\n' + column + ' unique values:', sorted(features[column].dropna().unique().tolist()))
    counts = features[column].value_counts(dropna=False)
    distribution = pd.DataFrame({'ICU_stays':counts,'percent':(counts/len(features)*100).round(2)})
    print(distribution.to_string())


Age calculation: (ICU INTIME - DOB) in seconds / (365.25 * 86400)
Upper cap: 90, a group representation, not an exact recovered age
Raw ages above 120 (anonymization diagnostic): 2239
Raw ages above 90 capped: 2239
Age min / median / max: 18.021689894035035 66.06536358911958 90.0

Missing values:
ICUSTAY_ID            0
age                   0
gender                0
admission_type        0
admission_location    0

gender unique values: ['F', 'M']
        ICU_stays  percent
gender                    
M           25587    56.54
F           19666    43.46

admission_type unique values: ['ELECTIVE', 'EMERGENCY', 'URGENT']
                ICU_stays  percent
admission_type                    
EMERGENCY           37572    83.03
ELECTIVE             6489    14.34
URGENT               1192     2.63

admission_location unique values: ['** INFO NOT AVAILABLE **', 'CLINIC REFERRAL/PREMATURE', 'EMERGENCY ROOM ADMIT', 'PHYS REFERRAL/NORMAL DELI', 'TRANSFER FROM HOSP/EXTRAM', 'TRANSFER FROM OTHER HE

In [3]:
expected_columns = ['ICUSTAY_ID','age','gender','admission_type','admission_location']
assert features.columns.tolist() == expected_columns
assert features.shape == (45253,5)
assert features.ICUSTAY_ID.is_unique
assert features.ICUSTAY_ID.equals(cohort.ICUSTAY_ID)
assert features.age.dropna().between(18,90).all()
# Confirm that only the requested source attributes and transformed age were used.
pd.testing.assert_series_equal(features.gender, merged.GENDER, check_names=False)
pd.testing.assert_series_equal(features.admission_type, merged.ADMISSION_TYPE, check_names=False)
pd.testing.assert_series_equal(features.admission_location, merged.ADMISSION_LOCATION, check_names=False)
pd.testing.assert_series_equal(features.age, age_raw.clip(upper=90), check_names=False)
assert {str(p):sha256(p) for p in protected} == protected_before
partial_path = output_path.with_suffix('.parquet.partial')
features.to_parquet(partial_path,index=False)
pd.testing.assert_frame_equal(pd.read_parquet(partial_path),features)
partial_path.replace(output_path)
assert {str(p):sha256(p) for p in protected} == protected_before
print('Saved:',output_path)
print('Final shape:', features.shape)
print('Unique ICUSTAY_ID:', features.ICUSTAY_ID.nunique())
print('Duplicate ICUSTAY_ID:', int(features.ICUSTAY_ID.duplicated().sum()))
print('Cohort membership and order exactly preserved.')
print('All protected cohort, lab, vital, output, intervention files and existing notebooks unchanged (SHA-256).')


Saved: /home/sengul/projects/icu-risk-prediction/data/processed/demographic_features.parquet
Final shape: (45253, 5)
Unique ICUSTAY_ID: 45253
Duplicate ICUSTAY_ID: 0
Cohort membership and order exactly preserved.
All protected cohort, lab, vital, output, intervention files and existing notebooks unchanged (SHA-256).


In [7]:
demographic_features_df = pd.read_parquet(
    "../data/processed/demographic_features.parquet"
)

print(demographic_features_df.shape)
print(demographic_features_df["ICUSTAY_ID"].nunique())
print(demographic_features_df.isna().sum())
demographic_features_df.head()

(45253, 5)
45253
ICUSTAY_ID            0
age                   0
gender                0
admission_type        0
admission_location    0
dtype: int64


,ICUSTAY_ID,age,gender,admission_type,admission_location
0,280836,65.984880,F,EMERGENCY,EMERGENCY ROOM ADMIT
1,206613,40.099828,M,EMERGENCY,EMERGENCY ROOM ADMIT
2,220345,80.078381,M,ELECTIVE,PHYS REFERRAL/NORMAL DELI
3,249196,45.686426,F,EMERGENCY,TRANSFER FROM HOSP/EXTRAM
4,210407,67.096182,M,EMERGENCY,TRANSFER FROM HOSP/EXTRAM
